In [0]:
from pyspark.sql.types import *

weather_schema = StructType([

    StructField("event_id", StringType()),

    StructField("ingestion_time", StringType()),

    StructField("city", StringType()),

    StructField("region", StringType()),

    StructField("country", StringType()),

    StructField("lat", DoubleType()),

    StructField("lon", DoubleType()),

    StructField("localtime", StringType()),

    StructField("temp_c", DoubleType()),

    StructField("condition_text", StringType()),

    StructField("condition_icon", StringType()),

    StructField("wind_kph", DoubleType()),

    StructField("wind_degree", IntegerType()),

    StructField("wind_dir", StringType()),

    StructField("pressure_mb", DoubleType()),

    StructField("humidity", IntegerType()),

    StructField("cloud", IntegerType()),

    StructField("feelslike_c", DoubleType()),

    StructField("uv", DoubleType()),

    StructField("pm2_5", DoubleType()),

    StructField("pm10", DoubleType()),

    StructField("epa_index", IntegerType()),

    # Forecast array
    StructField(
        "forecast",
        ArrayType(
            StructType([

                StructField("date", StringType()),

                StructField("maxtemp_c", DoubleType()),

                StructField("mintemp_c", DoubleType()),

                StructField("condition", StringType())
            ])
        )
    ),

    # Alerts array
    StructField(
        "alerts",
        ArrayType(
            StructType([

                StructField("headline", StringType()),

                StructField("severity", StringType()),

                StructField("description", StringType())
            ])
        )
    )
])

In [0]:
from pyspark.sql.functions import col, from_json

bronze_df = spark.readStream.table(
    "db_weather_streaming.bronze.bronze_weather"
)

parsed_df = bronze_df.withColumn(
    "parsed",
    from_json(col("body"), weather_schema)
)

In [0]:
%sql
CREATE OR REPLACE TABLE db_weather_streaming.silver.silver_weather (

  event_id STRING,

  ingestion_time TIMESTAMP,

  city STRING,

  region STRING,

  country STRING,

  lat DOUBLE,

  lon DOUBLE,

  localtime TIMESTAMP,

  temp_c DOUBLE,

  weather_condition STRING,

  wind_kph DOUBLE,

  wind_degree INT,

  wind_dir STRING,

  pressure_mb DOUBLE,

  humidity INT,

  cloud INT,

  feelslike_c DOUBLE,

  uv DOUBLE,

  pm2_5 DOUBLE,

  pm10 DOUBLE,

  epa_index INT

)
USING DELTA;

In [0]:
from pyspark.sql.functions import col, to_timestamp
silver_df = parsed_df.select(


    col("parsed.event_id").alias("event_id"),

    to_timestamp(
        col("parsed.ingestion_time")
    ).alias("ingestion_time"),

    col("parsed.city").alias("city"),

    col("parsed.region").alias("region"),

    col("parsed.country").alias("country"),

    col("parsed.lat").alias("lat"),

    col("parsed.lon").alias("lon"),

    to_timestamp(
        col("parsed.localtime")
    ).alias("localtime"),

    col("parsed.temp_c").alias("temp_c"),

    col("parsed.condition_text").alias("weather_condition"),

    col("parsed.wind_kph").alias("wind_kph"),

    col("parsed.wind_degree").alias("wind_degree"),

    col("parsed.wind_dir").alias("wind_dir"),

    col("parsed.pressure_mb").alias("pressure_mb"),

    col("parsed.humidity").alias("humidity"),

    col("parsed.cloud").alias("cloud"),

    col("parsed.feelslike_c").alias("feelslike_c"),

    col("parsed.uv").alias("uv"),

    col("parsed.pm2_5").alias("pm2_5"),

    col("parsed.pm10").alias("pm10"),

    col("parsed.epa_index").alias("epa_index")
)

In [0]:
silver_query = silver_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/db_weather_streaming/volumes/checkpoints/my_stream_silver") \
    .toTable("db_weather_streaming.silver.silver_weather")

In [0]:
dbutils.fs.rm("/Volumes/db_weather_streaming/volumes/checkpoints/my_stream_silver", recurse=True)

In [0]:
silver_query.stop()

In [0]:
%sql
select * from db_weather_streaming.bronze.bronze_weather

body
"{""event_id"": ""6b2be172-b198-4152-9e5c-fa3ac43013d4"", ""ingestion_time"": ""2026-05-09T05:43:48.384966"", ""city"": ""Chennai"", ""region"": ""Tamil Nadu"", ""country"": ""India"", ""lat"": 13.0833, ""lon"": 80.2833, ""localtime"": ""2026-05-09 11:13"", ""temp_c"": 30.1, ""condition_text"": ""Mist"", ""condition_icon"": ""//cdn.weatherapi.com/weather/64x64/day/143.png"", ""wind_kph"": 16.9, ""wind_degree"": 74, ""wind_dir"": ""ENE"", ""pressure_mb"": 1009.0, ""humidity"": 84, ""cloud"": 75, ""feelslike_c"": 34.0, ""uv"": 11.7, ""pm2_5"": 5.65, ""pm10"": 6.65, ""epa_index"": 1, ""forecast"": [{""date"": ""2026-05-09"", ""maxtemp_c"": 31.4, ""mintemp_c"": 28.5, ""condition"": ""Patchy rain nearby""}, {""date"": ""2026-05-10"", ""maxtemp_c"": 30.8, ""mintemp_c"": 28.0, ""condition"": ""Patchy rain nearby""}, {""date"": ""2026-05-11"", ""maxtemp_c"": 31.1, ""mintemp_c"": 28.1, ""condition"": ""Sunny""}], ""alerts"": []}"
"{""event_id"": ""7fc359de-ea09-4244-8b84-2212f127e837"", ""ingestion_time"": ""2026-05-09T05:44:19.815666"", ""city"": ""Chennai"", ""region"": ""Tamil Nadu"", ""country"": ""India"", ""lat"": 13.0833, ""lon"": 80.2833, ""localtime"": ""2026-05-09 11:13"", ""temp_c"": 30.1, ""condition_text"": ""Mist"", ""condition_icon"": ""//cdn.weatherapi.com/weather/64x64/day/143.png"", ""wind_kph"": 16.9, ""wind_degree"": 74, ""wind_dir"": ""ENE"", ""pressure_mb"": 1009.0, ""humidity"": 84, ""cloud"": 75, ""feelslike_c"": 34.0, ""uv"": 11.7, ""pm2_5"": 5.65, ""pm10"": 6.65, ""epa_index"": 1, ""forecast"": [{""date"": ""2026-05-09"", ""maxtemp_c"": 31.4, ""mintemp_c"": 28.5, ""condition"": ""Patchy rain nearby""}, {""date"": ""2026-05-10"", ""maxtemp_c"": 30.8, ""mintemp_c"": 28.0, ""condition"": ""Patchy rain nearby""}, {""date"": ""2026-05-11"", ""maxtemp_c"": 31.1, ""mintemp_c"": 28.1, ""condition"": ""Sunny""}], ""alerts"": []}"
"{""event_id"": ""ecb842a3-0a56-46cd-a493-0f1e4486b1d5"", ""ingestion_time"": ""2026-05-09T05:44:50.043024"", ""city"": ""Chennai"", ""region"": ""Tamil Nadu"", ""country"": ""India"", ""lat"": 13.0833, ""lon"": 80.2833, ""localtime"": ""2026-05-09 11:13"", ""temp_c"": 30.1, ""condition_text"": ""Mist"", ""condition_icon"": ""//cdn.weatherapi.com/weather/64x64/day/143.png"", ""wind_kph"": 16.9, ""wind_degree"": 74, ""wind_dir"": ""ENE"", ""pressure_mb"": 1009.0, ""humidity"": 84, ""cloud"": 75, ""feelslike_c"": 34.0, ""uv"": 11.7, ""pm2_5"": 5.65, ""pm10"": 6.65, ""epa_index"": 1, ""forecast"": [{""date"": ""2026-05-09"", ""maxtemp_c"": 31.4, ""mintemp_c"": 28.5, ""condition"": ""Patchy rain nearby""}, {""date"": ""2026-05-10"", ""maxtemp_c"": 30.8, ""mintemp_c"": 28.0, ""condition"": ""Patchy rain nearby""}, {""date"": ""2026-05-11"", ""maxtemp_c"": 31.1, ""mintemp_c"": 28.1, ""condition"": ""Sunny""}], ""alerts"": []}"
"{""event_id"": ""87f89299-6350-4230-b5b5-5aa649a4dbe2"", ""ingestion_time"": ""2026-05-09T05:45:20.248622"", ""city"": ""Chennai"", ""region"": ""Tamil Nadu"", ""country"": ""India"", ""lat"": 13.0833, ""lon"": 80.2833, ""localtime"": ""2026-05-09 11:13"", ""temp_c"": 30.1, ""condition_text"": ""Mist"", ""condition_icon"": ""//cdn.weatherapi.com/weather/64x64/day/143.png"", ""wind_kph"": 16.9, ""wind_degree"": 74, ""wind_dir"": ""ENE"", ""pressure_mb"": 1009.0, ""humidity"": 84, ""cloud"": 75, ""feelslike_c"": 34.0, ""uv"": 11.7, ""pm2_5"": 5.65, ""pm10"": 6.65, ""epa_index"": 1, ""forecast"": [{""date"": ""2026-05-09"", ""maxtemp_c"": 31.4, ""mintemp_c"": 28.5, ""condition"": ""Patchy rain nearby""}, {""date"": ""2026-05-10"", ""maxtemp_c"": 30.8, ""mintemp_c"": 28.0, ""condition"": ""Patchy rain nearby""}, {""date"": ""2026-05-11"", ""maxtemp_c"": 31.1, ""mintemp_c"": 28.1, ""condition"": ""Sunny""}], ""alerts"": []}"
"{""event_id"": ""3acf212e-4a5f-42db-b5f8-4f76aed2ec3f"", ""ingestion_time"": ""2026-05-09T05:45:50.428283"", ""city"": ""Chennai"", ""region"": ""Tamil Nadu"", ""country"": ""India"", ""lat"": 13.0833, ""lon"": 80.2833, ""localtime"": ""2026-05-09 11:13"

In [0]:
%sql
select * from db_weather_streaming.silver.silver_weather

event_id,ingestion_time,city,region,country,lat,lon,localtime,temp_c,weather_condition,wind_kph,wind_degree,wind_dir,pressure_mb,humidity,cloud,feelslike_c,uv,pm2_5,pm10,epa_index
6b2be172-b198-4152-9e5c-fa3ac43013d4,2026-05-09T05:43:48.384966Z,Chennai,Tamil Nadu,India,13.0833,80.2833,2026-05-09T11:13:00Z,30.1,Mist,16.9,74,ENE,1009.0,84,75,34.0,11.7,5.65,6.65,1
7fc359de-ea09-4244-8b84-2212f127e837,2026-05-09T05:44:19.815666Z,Chennai,Tamil Nadu,India,13.0833,80.2833,2026-05-09T11:13:00Z,30.1,Mist,16.9,74,ENE,1009.0,84,75,34.0,11.7,5.65,6.65,1
ecb842a3-0a56-46cd-a493-0f1e4486b1d5,2026-05-09T05:44:50.043024Z,Chennai,Tamil Nadu,India,13.0833,80.2833,2026-05-09T11:13:00Z,30.1,Mist,16.9,74,ENE,1009.0,84,75,34.0,11.7,5.65,6.65,1
87f89299-6350-4230-b5b5-5aa649a4dbe2,2026-05-09T05:45:20.248622Z,Chennai,Tamil Nadu,India,13.0833,80.2833,2026-05-09T11:13:00Z,30.1,Mist,16.9,74,ENE,1009.0,84,75,34.0,11.7,5.65,6.65,1
3acf212e-4a5f-42db-b5f8-4f76aed2ec3f,2026-05-09T05:45:50.428283Z,Chennai,Tamil Nadu,India,13.0833,80.2833,2026-05-09T11:13:00Z,30.1,Mist,16.9,74,ENE,1009.0,84,75,34.0,11.7,5.65,6.65,1


forecast table details

In [0]:
%sql
CREATE OR REPLACE TABLE db_weather_streaming.silver.silver_forecast (

  event_id STRING,

  ingestion_time TIMESTAMP,

  city STRING,

  forecast_date DATE,

  max_temp_c DOUBLE,

  min_temp_c DOUBLE,

  forecast_condition STRING

)
USING DELTA;

In [0]:
from pyspark.sql.functions import explode, col, to_timestamp, to_date

forecast_df = parsed_df.select(

    col("parsed.event_id").alias("event_id"),

    to_timestamp(
        col("parsed.ingestion_time")
    ).alias("ingestion_time"),

    col("parsed.city").alias("city"),

    explode(
        col("parsed.forecast")
    ).alias("forecast_data")
)

In [0]:
silver_forecast_df = forecast_df.select(

    col("event_id"),

    col("ingestion_time"),

    col("city"),

    to_date(
        col("forecast_data.date")
    ).alias("forecast_date"),

    col("forecast_data.maxtemp_c").alias("max_temp_c"),

    col("forecast_data.mintemp_c").alias("min_temp_c"),

    col("forecast_data.condition").alias("forecast_condition")
)

In [0]:
forecast_query = silver_forecast_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation","/Volumes/db_weather_streaming/volumes/checkpoints/silver_forecast") \
    .toTable("db_weather_streaming.silver.silver_forecast"
    )

In [0]:
forecast_query.stop()

In [0]:
%sql
select * from db_weather_streaming.silver.silver_forecast

event_id,ingestion_time,city,forecast_date,max_temp_c,min_temp_c,forecast_condition
6b2be172-b198-4152-9e5c-fa3ac43013d4,2026-05-09T05:43:48.384966Z,Chennai,2026-05-09,31.4,28.5,Patchy rain nearby
6b2be172-b198-4152-9e5c-fa3ac43013d4,2026-05-09T05:43:48.384966Z,Chennai,2026-05-10,30.8,28.0,Patchy rain nearby
6b2be172-b198-4152-9e5c-fa3ac43013d4,2026-05-09T05:43:48.384966Z,Chennai,2026-05-11,31.1,28.1,Sunny
7fc359de-ea09-4244-8b84-2212f127e837,2026-05-09T05:44:19.815666Z,Chennai,2026-05-09,31.4,28.5,Patchy rain nearby
7fc359de-ea09-4244-8b84-2212f127e837,2026-05-09T05:44:19.815666Z,Chennai,2026-05-10,30.8,28.0,Patchy rain nearby
7fc359de-ea09-4244-8b84-2212f127e837,2026-05-09T05:44:19.815666Z,Chennai,2026-05-11,31.1,28.1,Sunny
ecb842a3-0a56-46cd-a493-0f1e4486b1d5,2026-05-09T05:44:50.043024Z,Chennai,2026-05-09,31.4,28.5,Patchy rain nearby
ecb842a3-0a56-46cd-a493-0f1e4486b1d5,2026-05-09T05:44:50.043024Z,Chennai,2026-05-10,30.8,28.0,Patchy rain nearby
ecb842a3-0a56-46cd-a493-0f1e4486b1d5,2026-05-09T05:44:50.043024Z,Chennai,2026-05-11,31.1,28.1,Sunny
87f89299-6350-4230-b5b5-5aa649a4dbe2,2026-05-09T05:45:20.248622Z,Chennai,2026-05-09,31.4,28.5,Patchy rain nearby
